# 🧰 L5 — Connecter des outils MCP à un agent LangChain

> ⏱️ **Durée indicative : 30 à 45 minutes**  
> 🎓 **Niveau : débutant** — aucune connaissance préalable de MCP n'est nécessaire.

Dans la leçon précédente, nous avons donné une fonction Python à un agent. Ici, nous allons découvrir comment lui donner un outil fourni par **un autre programme**, grâce au **Model Context Protocol (MCP)**.

📚 Dès maintenant, gardez sous la main la [documentation MCP officielle de LangChain](https://docs.langchain.com/oss/python/langchain/mcp).

## 🎯 Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- expliquer simplement la différence entre un **outil local** et un **outil MCP** ;
- distinguer le **serveur**, l'**adaptateur** et le **transport** ;
- découvrir les outils exposés par un serveur MCP et lire leur schéma ;
- connecter ces outils à un agent créé avec `create_agent` ;
- repérer précisément ce que décide Mistral et ce qu'exécute réellement Python.

### 🧭 Notre signalétique

🎯 objectif · 🧠 intuition · 🗺️ schéma mental · 🛠️ construction · 🔮 prédiction · ▶️ exécution · 👀 observation · ⚠️ piège · 🧪 exercice · ✅ correction · 📚 documentation

## 🗺️ La carte du voyage

```text
Question de l'utilisateur
          │
          ▼
🤖 Agent LangChain ──► 🧠 Mistral choisit un outil et ses arguments
          │
          ▼
🔌 Adaptateur LangChain MCP
          │  transport stdio (messages via entrée/sortie standard)
          ▼
🧰 Serveur MCP time ──► 🕒 exécute réellement l'outil
          │
          ▼
Résultat renvoyé à Mistral, puis réponse finale
```

🧠 **Analogie ELI5 adulte :** l'agent est un maître d'hôtel, l'adaptateur est l'interprète, le serveur MCP est la cuisine et le transport `stdio` est le passe-plat. Le maître d'hôtel choisit quoi commander ; la cuisine réalise vraiment l'action.

## 🔐 1. Préparer Mistral sans exposer de secret

Ce notebook utilise uniquement les variables déjà présentes dans le **processus Jupyter** :

- `MISTRAL_API_KEY` ;
- `MISTRAL_SERVER_URL`.

Il ne lit aucun fichier `.env` et n'affiche jamais les valeurs. L'URL peut se terminer ou non par `/v1` ; nous la normalisons une seule fois.

📚 `ChatMistralAI` et ses paramètres sont décrits dans l'[intégration officielle LangChain–Mistral](https://docs.langchain.com/oss/python/integrations/chat/mistralai).

### 🔮 Pause prédiction

Si `MISTRAL_SERVER_URL` vaut `https://mon-serveur.example/v1/`, quelle URL voulons-nous transmettre au client ?

<details><summary>💡 Voir la réponse</summary>

`https://mon-serveur.example/v1` : un seul suffixe `/v1`, sans `/` final.
</details>

In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())
import os

from langchain_mistralai import ChatMistralAI


def normalize_mistral_endpoint(raw_url: str) -> str:
    """Return the API base URL expected by ChatMistralAI."""
    endpoint = raw_url.strip().rstrip("/")
    # Certains portails fournissent l'URL complète ; ChatMistralAI ajoute lui-même
    # /chat/completions à l'endpoint de base.
    endpoint = endpoint.removesuffix("/chat/completions")
    if not endpoint.endswith("/v1"):
        endpoint = f"{endpoint}/v1"
    return endpoint


required_variables = ("MISTRAL_API_KEY", "MISTRAL_SERVER_URL")
missing_variables = [name for name in required_variables if not os.environ.get(name)]
if missing_variables:
    raise RuntimeError(
        "❌ Variables manquantes dans le processus Jupyter : "
        + ", ".join(missing_variables)
        + ". Définissez-les avant de lancer Jupyter, puis redémarrez le kernel."
    )

mistral_endpoint = normalize_mistral_endpoint(os.environ["MISTRAL_SERVER_URL"])

# 📚 API officielle : https://docs.langchain.com/oss/python/integrations/chat/mistralai
mistral_model = ChatMistralAI(
    model="mistral-medium-latest",
    temperature=0,
    api_key=os.environ["MISTRAL_API_KEY"],
    endpoint=mistral_endpoint,
)

print("✅ Configuration Mistral détectée — aucune valeur n'est affichée.")

✅ Configuration Mistral détectée — aucune valeur n'est affichée.


## 🧰 2. Point de comparaison : un outil Python local

Un **outil local** est une fonction qui vit dans le même programme que notre agent. Le décorateur [`@tool`](https://docs.langchain.com/oss/python/langchain/tools) transforme sa signature et sa docstring en une fiche lisible par le modèle : nom, description et schéma des arguments.

Ici, Python connaît déjà les fuseaux horaires grâce à sa bibliothèque standard. Aucun serveur externe n'est nécessaire.

### 🔮 Pause prédiction

Qui exécutera réellement `local_current_time` : Mistral ou Python ?

➡️ Formulez votre réponse avant d'exécuter les deux cellules suivantes.

In [2]:
from datetime import datetime
from zoneinfo import ZoneInfo

from langchain.tools import tool


# 📚 @tool : https://docs.langchain.com/oss/python/langchain/tools
@tool
def local_current_time(timezone: str) -> str:
    """Return the current time for an IANA timezone such as Europe/Paris."""
    now = datetime.now(ZoneInfo(timezone))
    return now.isoformat(timespec="seconds")

In [3]:
import json

print("Nom :", local_current_time.name)
print("Description :", local_current_time.description)
print("Schéma :")
print(json.dumps(local_current_time.args_schema.model_json_schema(), indent=2, ensure_ascii=False))
print("Résultat direct :", local_current_time.invoke({"timezone": "Europe/Paris"}))

Nom : local_current_time
Description : Return the current time for an IANA timezone such as Europe/Paris.
Schéma :
{
  "description": "Return the current time for an IANA timezone such as Europe/Paris.",
  "properties": {
    "timezone": {
      "title": "Timezone",
      "type": "string"
    }
  },
  "required": [
    "timezone"
  ],
  "title": "local_current_time",
  "type": "object"
}


Résultat direct : 2026-09-02T10:36:57+02:00


### 👀 Ce qu'il faut observer

- LangChain a fabriqué le **contrat** de l'outil à partir du code Python.
- Le paramètre `timezone` apparaît dans un schéma JSON.
- Lors de l'appel direct, c'est bien **Python** qui exécute la fonction.
- Mistral n'a encore rien décidé : nous n'avons pas appelé le modèle.

🧠 Un outil MCP offrira le même type de contrat à LangChain, mais son code vivra dans un autre processus.

## 🔌 3. MCP en trois pièces faciles à distinguer

### 🧰 Le serveur MCP

Le serveur est le programme qui **possède et exécute** les capacités. Nous utiliserons le serveur officiel [`mcp-server-time`](https://pypi.org/project/mcp-server-time/), qui expose notamment `get_current_time` et `convert_time`.

### 🔌 L'adaptateur LangChain

[`MultiServerMCPClient`](https://docs.langchain.com/oss/python/langchain/mcp) se connecte au serveur, découvre ses outils et les convertit en outils compris par LangChain. Malgré son nom, il fonctionne aussi avec un seul serveur.

### 🚇 Le transport `stdio`

Le transport est la route empruntée par les messages. Avec `stdio`, LangChain lance un sous-processus et échange des messages structurés via son entrée et sa sortie standard. Ce n'est ni une page web ni un appel HTTP.

> 🧠 **À retenir :** MCP standardise la conversation entre programmes ; il ne remplace ni LangChain ni le modèle.

## ⚙️ 4. Préparer `uvx` et le cas Windows

[`uvx`](https://docs.astral.sh/uv/guides/tools/) lance un outil Python dans un environnement isolé. Nous épinglons `mcp-server-time==2026.8.18` pour que tous les participants utilisent la même version. Cette version déclare elle-même `mcp>=1.29.0,<2` : aucun ajout manuel de `mcp<2` n'est nécessaire.

Sous Windows, certains kernels Jupyter remplacent `sys.stderr` par un objet sans descripteur de fichier. Le SDK MCP en a besoin pour lancer le sous-processus. Le contournement ci-dessous ne s'applique **que si ce problème est réellement détecté**, et avant l'import du client MCP.

### 🔮 Pause prédiction

Que doit faire le notebook si `uvx` est absent ?

A. Continuer et échouer plus loin avec une erreur obscure.  
B. S'arrêter immédiatement avec une instruction d'installation claire.

✅ La réponse attendue est **B**.

In [4]:
import io
import shutil
import sys

uvx_path = shutil.which("uvx")
if uvx_path is None:
    raise RuntimeError(
        "❌ uvx est introuvable. Installez uv depuis https://docs.astral.sh/uv/, "
        "fermez puis rouvrez le terminal, et redémarrez le kernel Jupyter."
    )


def stderr_has_fileno() -> bool:
    """Return whether the current stderr can be passed to a subprocess."""
    try:
        sys.stderr.fileno()
    except (AttributeError, io.UnsupportedOperation, OSError, ValueError):
        return False
    return True


if sys.platform == "win32" and not stderr_has_fileno():
    sys.stderr = sys.__stderr__
    print("⚙️ Compatibilité Windows activée pour le sous-processus MCP.")
else:
    print("✅ Aucun contournement stderr nécessaire.")

# 📚 L'import vient après le contrôle Windows afin que le SDK capture le bon stderr.
# Documentation officielle : https://docs.langchain.com/oss/python/langchain/mcp
from langchain_mcp_adapters.client import MultiServerMCPClient

⚙️ Compatibilité Windows activée pour le sous-processus MCP.


## 🛠️ 5. Décrire la connexion au serveur

Nous donnons maintenant à l'adaptateur une petite fiche de connexion :

- `command` : le programme à lancer ;
- `args` : ses arguments ;
- `transport` : la manière d'échanger les messages.

À ce stade, nous ne choisissons encore aucun outil : nous indiquons seulement **où se trouve le serveur** et **comment lui parler**.

In [5]:
MCP_SERVER_TIME_VERSION = "2026.8.18"

# 📚 MultiServerMCPClient : https://docs.langchain.com/oss/python/langchain/mcp
mcp_client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": uvx_path,
            "args": [
                "--from",
                f"mcp-server-time=={MCP_SERVER_TIME_VERSION}",
                "mcp-server-time",
            ],
        }
    }
)

print("✅ Configuration du serveur MCP prête.")

✅ Configuration du serveur MCP prête.


## 🔍 6. Découvrir les outils au lieu de les deviner

### 🔮 Pause prédiction

Combien d'outils le serveur expose-t-il ? Quels arguments attendent-ils ?

Ne cherchez pas dans le code du serveur : demandons-le directement via MCP. C'est précisément l'un des bénéfices du protocole.

In [6]:
# get_tools() démarre le serveur, effectue la négociation MCP et transforme
# ses outils en outils LangChain. Le client est sans état par défaut.
# 📚 https://docs.langchain.com/oss/python/langchain/mcp
mcp_tools = await mcp_client.get_tools()


def schema_outil(outil) -> dict:
    """Retourne le schéma JSON d'un outil, qu'il soit local ou MCP.

    Un outil local (@tool) expose un modèle Pydantic doté de model_json_schema(),
    tandis qu'un outil MCP expose déjà son schéma directement sous forme de dict.
    """
    args_schema = outil.args_schema
    if args_schema is None:
        return {}
    if isinstance(args_schema, dict):
        return args_schema
    return args_schema.model_json_schema()


print(f"🧰 {len(mcp_tools)} outil(s) MCP découvert(s).")
for discovered_tool in mcp_tools:
    print()
    print(f"🔧 {discovered_tool.name}")
    print(f"Description : {discovered_tool.description}")
    print("Schéma :")
    print(json.dumps(schema_outil(discovered_tool), indent=2, ensure_ascii=False))

🧰 2 outil(s) MCP découvert(s).

🔧 get_current_time
Description : Get current time in a specific timezone
Schéma :
{
  "type": "object",
  "properties": {
    "timezone": {
      "type": "string",
      "description": "IANA timezone name (e.g., 'America/New_York', 'Europe/London'). Use 'Europe/Paris' as local timezone if no timezone provided by the user."
    }
  },
  "required": [
    "timezone"
  ]
}

🔧 convert_time
Description : Convert time between timezones
Schéma :
{
  "type": "object",
  "properties": {
    "source_timezone": {
      "type": "string",
      "description": "Source IANA timezone name (e.g., 'America/New_York', 'Europe/London'). Use 'Europe/Paris' as local timezone if no source timezone provided by the user."
    },
    "time": {
      "type": "string",
      "description": "Time to convert in 24-hour format (HH:MM)"
    },
    "target_timezone": {
      "type": "string",
      "description": "Target IANA timezone name (e.g., 'Asia/Tokyo', 'America/San_Francisco'). 

### 👀 Ce qu'il faut observer

Vous devriez découvrir notamment :

- `get_current_time`, qui attend un fuseau IANA comme `America/Los_Angeles` ;
- `convert_time`, qui convertit une heure entre deux fuseaux.

🔍 Le schéma joue le même rôle que pour notre outil local. La grande différence est son **origine** : il a été annoncé par le serveur pendant la connexion MCP.

⚠️ `MultiServerMCPClient` est sans état par défaut : chaque appel d'outil ouvre une nouvelle session MCP, exécute l'action, puis la ferme. C'est suffisant pour ce serveur d'heure.

## ▶️ 7. Appeler un outil MCP directement

Avant d'ajouter Mistral, isolons une seule pièce du système. Nous appelons directement l'outil `get_current_time`.

### 🔮 Pause prédiction

Le résultat proviendra-t-il de LangChain, de Mistral ou du serveur MCP ?

In [7]:
get_current_time_tool = next(
    tool for tool in mcp_tools if tool.name == "get_current_time"
)

san_francisco_time = await get_current_time_tool.ainvoke(
    {"timezone": "America/Los_Angeles"}
)
print(san_francisco_time)

[{'type': 'text', 'text': '{\n  "timezone": "America/Los_Angeles",\n  "datetime": "2026-09-02T01:37:10-07:00",\n  "day_of_week": "Wednesday",\n  "is_dst": true\n}', 'id': 'lc_5676e303-9fce-440f-9c75-b5d97ad87c87'}]


### 👀 Lecture de la sortie

Le **serveur MCP** a calculé l'heure. L'adaptateur a transporté la requête et converti le résultat. Mistral n'a toujours pas été appelé.

Cette étape est une technique de diagnostic très utile : si l'appel direct fonctionne mais pas l'agent, le serveur et le transport ne sont probablement pas la cause.

## 🤖 8. Confier le choix de l'outil à l'agent

[`create_agent`](https://docs.langchain.com/oss/python/langchain/agents) orchestre maintenant la boucle complète :

1. LangChain transmet à Mistral la question et les fiches des outils.
2. Mistral choisit éventuellement un outil et produit ses arguments.
3. LangChain demande à l'outil MCP de s'exécuter.
4. Le serveur time calcule le résultat.
5. LangChain renvoie ce résultat à Mistral pour formuler la réponse finale.

📚 Ce cycle correspond aux [cinq étapes du function calling documentées par Mistral](https://docs.mistral.ai/studio/conversations/function-calling). Le modèle **propose** l'appel ; il n'exécute pas lui-même le programme.

In [8]:
from langchain.agents import create_agent

# 📚 create_agent : https://docs.langchain.com/oss/python/langchain/agents
agent_with_mcp = create_agent(
    model=mistral_model,
    tools=mcp_tools,
    system_prompt=(
        "Tu es un assistant précis. Pour toute heure actuelle ou conversion de fuseau, "
        "utilise un outil disponible au lieu de deviner. Réponds en français."
    ),
)

### 🔮 Pause prédiction

Pour « Quelle heure est-il à San Francisco ? », notez :

- le nom de l'outil que Mistral devrait choisir ;
- l'argument de fuseau probable ;
- l'ordre attendu des messages : humain → assistant avec appel → outil → assistant final.

In [9]:
result = await agent_with_mcp.ainvoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Quelle heure est-il maintenant à San Francisco ?",
            }
        ]
    }
)

for index, message in enumerate(result["messages"], start=1):
    print(f"\n--- Message {index} · {type(message).__name__} ---")
    tool_calls = getattr(message, "tool_calls", None)
    if tool_calls:
        print("🧠 Appel(s) décidé(s) par Mistral :")
        print(json.dumps(tool_calls, indent=2, ensure_ascii=False))
    message.pretty_print()


--- Message 1 · HumanMessage ---
================================ Human Message =================================

Quelle heure est-il maintenant à San Francisco ?

--- Message 2 · AIMessage ---
🧠 Appel(s) décidé(s) par Mistral :
[
  {
    "name": "get_current_time",
    "args": {
      "timezone": "America/Los_Angeles"
    },
    "id": "VNN7eN2q8",
    "type": "tool_call"
  }
]
================================== Ai Message ==================================
Tool Calls:
  get_current_time (VNN7eN2q8)
 Call ID: VNN7eN2q8
  Args:
    timezone: America/Los_Angeles

--- Message 3 · ToolMessage ---
================================= Tool Message =================================
Name: get_current_time

[{'type': 'text', 'text': '{\n  "timezone": "America/Los_Angeles",\n  "datetime": "2026-09-02T01:37:19-07:00",\n  "day_of_week": "Wednesday",\n  "is_dst": true\n}', 'id': 'lc_ee870331-8767-4265-80fb-46b57ff8bf0d'}]

--- Message 4 · AIMessage ---
================================== Ai Message =

### 🔍 Anatomie de ce qui vient de se passer

| Acteur | Responsabilité observée |
|---|---|
| 🧠 Mistral | choisit `get_current_time`, génère le fuseau et rédige la réponse |
| 🔗 LangChain | maintient les messages et orchestre l'aller-retour |
| 🔌 Adaptateur MCP | convertit l'outil MCP en outil LangChain et transporte l'appel |
| 🧰 Serveur time | exécute réellement le calcul de l'heure |
| 🐍 Python | fait tourner le client, l'agent et le sous-processus |

⚠️ Si aucun appel d'outil n'apparaît, ne confondez pas cela avec une panne MCP : le modèle peut avoir choisi de répondre seul. La consigne système réduit ce risque, mais le choix appartient toujours au modèle.

## ⚖️ Outil local ou outil MCP ?

| Question | Outil local | Outil MCP |
|---|---|---|
| Où vit le code ? | Dans ce notebook/processus | Dans un serveur séparé |
| Qui décrit le schéma ? | Le décorateur `@tool` | Le serveur pendant la découverte |
| Faut-il un transport ? | Non | Oui : ici `stdio` |
| Déploiement indépendant ? | Difficile | Oui |
| Usage idéal | Petite logique propre à l'application | Capacité partagée entre plusieurs clients |

🧠 MCP devient particulièrement intéressant lorsqu'une même capacité doit servir plusieurs assistants, langages ou applications. Pour une fonction de trois lignes utilisée une seule fois, un outil local reste souvent plus simple.

## 🧪 Micro-exercice — Quelle heure est-il à Tokyo ?

Complétez la cellule suivante pour appeler **directement** `get_current_time_tool` avec le fuseau IANA de Tokyo.

### ✅ Critères de réussite

- vous utilisez `await` et `.ainvoke(...)` ;
- l'argument s'appelle `timezone` ;
- le fuseau vaut `Asia/Tokyo` ;
- la sortie mentionne Tokyo et contient une date/heure.

In [10]:
# TODO 🧪 Décommentez et complétez les deux lignes.
# tokyo_time = await get_current_time_tool.ainvoke({"timezone": "..."})
# print(tokyo_time)

<details>
<summary>✅ Afficher la correction</summary>

```python
tokyo_time = await get_current_time_tool.ainvoke(
    {"timezone": "Asia/Tokyo"}
)
print(tokyo_time)
```

🔍 Ici, aucun modèle n'intervient : Python appelle l'outil LangChain, l'adaptateur dialogue en `stdio`, puis le serveur MCP calcule l'heure.
</details>

## ⚠️ Pièges fréquents et diagnostic

- **`uvx` introuvable** : installez `uv`, rouvrez le terminal et redémarrez le kernel.
- **`Connection closed`** : exécutez d'abord la cellule de découverte ; elle sépare un problème serveur/transport d'un problème de modèle.
- **Mauvais fuseau** : utilisez un nom IANA (`Europe/Paris`), pas seulement `Paris`.
- **Confusion sur l'exécution** : Mistral génère une intention et des arguments ; le serveur exécute l'action.
- **État MCP supposé** : le client est sans état par défaut. Une session persistante doit être demandée explicitement si un futur serveur en a besoin.
- **Windows/Jupyter** : le contournement `stderr` n'est appliqué que lorsque `fileno()` manque ; ne le copiez pas aveuglément dans une application classique.

## ✅ Ce que vous savez maintenant

Vous savez désormais que :

1. un outil local et un outil MCP présentent tous deux un contrat structuré à LangChain ;
2. le serveur possède la capacité, l'adaptateur traduit et le transport déplace les messages ;
3. `MultiServerMCPClient.get_tools()` permet de découvrir les outils et leurs schémas ;
4. Mistral choisit l'outil, mais Python et le serveur l'exécutent réellement ;
5. un appel direct de l'outil aide à diagnostiquer chaque couche séparément.

## 🧭 Transition vers L6

Notre agent sait maintenant agir grâce à des outils externes. Mais si nous lui posons une deuxième question, se souviendra-t-il de la première ? Dans **L6 — Memory**, nous ajouterons une mémoire courte durée et vérifierons l'isolation entre conversations.

## 📚 Documentation officielle

- [LangChain — Model Context Protocol (MCP)](https://docs.langchain.com/oss/python/langchain/mcp)
- [LangChain — Tools](https://docs.langchain.com/oss/python/langchain/tools)
- [LangChain — Agents et `create_agent`](https://docs.langchain.com/oss/python/langchain/agents)
- [LangChain — Intégration `ChatMistralAI`](https://docs.langchain.com/oss/python/integrations/chat/mistralai)
- [Mistral AI — Les cinq étapes du function calling](https://docs.mistral.ai/studio/conversations/function-calling)
- [Serveur MCP officiel `mcp-server-time` sur PyPI](https://pypi.org/project/mcp-server-time/)
- [Code source officiel du serveur time](https://github.com/modelcontextprotocol/servers/tree/main/src/time)
- [uv — Exécuter des outils avec `uvx`](https://docs.astral.sh/uv/guides/tools/)